In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your api key"

In [71]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai faiss-cpu python-dotenv sentence-transformers

In [72]:
from youtube_transcript_api import YouTubeTranscriptApi , TranscriptsDisabled
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
import time

# Step 1a - Indexing (Document Ingestion)

In [73]:
video_id = "pJdMxwXBsk0" #only id, not full url
try:

    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(
        video_id,
        languages=["hi"],
    )

    transcript_data = transcript_list.to_raw_data()

    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

    print("First 3 transcript entries:")
    print(transcript_data[:3])

except TranscriptsDisabled:
    print("No captions available for this video.")

हाय गाइस, माय नेम इज नितेश एंड यू वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। और आज का वीडियो का टॉपिक है रिट्रीवर्स जो कि एक बहुत इंपॉर्टेंट टॉपिक है। अगर आप रैग की बात करो। अगर आप एक रैग बेस्ड एप्लीकेशन बनाना चाहते हो तो वहां पे रिट्रीवर एक बहुत इंपॉर्टेंट कंपोनेंट है। इनफैक्ट अ फ्यूचर में जब आप थोड़े एडवांस्ड रैग सिस्टम्स बनाओगे तो वहां पे अलग-अलग टाइप के रिट्रीवर्स के साथ आप काम करोगे तो उस सेंस में ये जो पर्टिकुलर वीडियो है बहुत इंपॉर्टेंट है। एंड आई वुड लाइक कि आप इस वीडियो को एंड टू एंड देखो। सो आज की वीडियो में मैं आपको नॉट ओनली समझाऊंगा कि रिट्रीवर्स क्या होते हैं? उनकी जरूरत क्या है? बट एट द सेम टाइम मैं आपको अलग-अलग टाइप के रिट्रीवर्स के बारे में भी बताऊंगा और जो कोड है वो लिख के दिखाऊंगा। ठीक है? सो या लेट्स स्टार्ट द वीडियो। सो गाइस वीडियो को शुरू करने के पहले मैं आपको एक क्विक रिककैप देना चाहूंगा कि पिछले तीन-चार वीडियोस से हम लोग इस प्लेलिस्ट में क्या कर रहे हैं। सो इनिशियली तो हम इस प्लेलिस्ट में लंगचेन के फंडामेंटल्स कवर 

In [74]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='हाय गाइस, माय नेम इज नितेश एंड यू वेलकम', start=0.32, duration=4.56), FetchedTranscriptSnippet(text='टू माय YouTube चैनल। इस वीडियो में भी हम', start=2.639, duration=4.001), FetchedTranscriptSnippet(text='लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू', start=4.88, duration=4.52), FetchedTranscriptSnippet(text='करेंगे। और आज का वीडियो का टॉपिक है', start=6.64, duration=5.039), FetchedTranscriptSnippet(text='रिट्रीवर्स जो कि एक बहुत इंपॉर्टेंट', start=9.4, duration=4.6), FetchedTranscriptSnippet(text='टॉपिक है। अगर आप रैग की बात करो। अगर आप', start=11.679, duration=4.801), FetchedTranscriptSnippet(text='एक रैग बेस्ड एप्लीकेशन बनाना चाहते हो तो', start=14.0, duration=4.56), FetchedTranscriptSnippet(text='वहां पे रिट्रीवर एक बहुत इंपॉर्टेंट', start=16.48, duration=4.879), FetchedTranscriptSnippet(text='कंपोनेंट है। इनफैक्ट अ फ्यूचर में जब आप', start=18.56, duration=5.52), FetchedTranscriptSnippet(text='थोड़े एडवांस्ड रैग सिस्टम्स बनाओगे तो'

# Step 1b - Indexing(Text Splitting)

In [75]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [76]:
len(chunks)

110

In [77]:
chunks[100]

Document(metadata={}, page_content='सकते हो। बस यहां पे आपको अपना मॉडल डाल देना है। ठीक है? और इस स्टेप में हमने अपना रिट्रीवर फॉर्म कर लिया। अब मान लो हमारा क्वेश्चन है व्हाट इज़ फोटोसिंथेसिस? हमने क्या किया? कंप्रेशन रिट्रीवर डॉट इनवोक को कॉल किया और अपनी क्वेरी पास की जिसने पलट के हमें रिजल्ट्स दिए और अब हम रिजल्ट्स को प्रिंट कर रहे हैं। एंड यू कैन सी गाइस यह रहे हमारे रिजल्ट्स। आप नोटिस करोगे कि भले ही हमारे सारे के सारे डॉक्यूमेंट्स एक-एक पैराग्राफ लॉन्ग है बट हमें जो पलट के आंसर्स मिल रहे हैं वो बहुत शॉर्ट वन सेंटेंस आंसर्स')

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [79]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Define batch size for processing documents to stay within rate limits
BATCH_SIZE = 10  # Process 10 documents at a time
SLEEP_TIME = 5   # Sleep for 5 seconds between batches

# Initialize an empty FAISS vector store
vector_store = None

for i in range(0, len(chunks), BATCH_SIZE):
    batch_chunks = chunks[i:i + BATCH_SIZE]
    print(f"Processing batch {i // BATCH_SIZE + 1} of {len(chunks) // BATCH_SIZE + (1 if len(chunks) % BATCH_SIZE else 0)}...")

    # Embed the batch of documents
    if vector_store is None:
        vector_store = FAISS.from_documents(batch_chunks, embeddings)
    else:
        vector_store.add_documents(batch_chunks) # Removed 'embeddings' argument here

    # Sleep to respect rate limits, but only if there are more batches to process
    if i + BATCH_SIZE < len(chunks):
        print(f"Sleeping for {SLEEP_TIME} seconds to respect API rate limits...")
        time.sleep(SLEEP_TIME)

print("FAISS vector store created successfully!")

Processing batch 1 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 2 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 3 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 4 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 5 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 6 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 7 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 8 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 9 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 10 of 11...
Sleeping for 5 seconds to respect API rate limits...
Processing batch 11 of 11...
FAISS vector store created successfully!


In [80]:
vector_store.index_to_docstore_id

{0: '8836cdbc-cb7c-4625-b850-07e897e347aa',
 1: 'c50191a1-7661-4605-8232-f0ea27441c56',
 2: '0b724330-4c47-40cb-974d-ac475c449293',
 3: 'c9cbdf28-9f2d-4bf3-ac54-c3ded1c3f417',
 4: '8ac32b0b-b2c7-4fb2-a05e-e210f76f34f6',
 5: 'c6f726bd-8bfd-4d84-8600-06df9d34a8f4',
 6: '0d6bf6ca-218d-437d-919a-016fe88f1af0',
 7: 'b3f41ba1-b0b8-40b7-afdf-dccd05d68cb3',
 8: '7ddacd0b-3314-4afd-aba1-11432f5ae0ee',
 9: 'ad0896e0-9808-4e2b-8b62-5e9541d00cba',
 10: '18a22386-f86a-4cca-9dc4-c890590cb3a3',
 11: '7896207a-5513-4cbc-bb8f-d014c524023a',
 12: '7a5546f6-b12a-4234-bcee-378f3363b742',
 13: '7dc2dc95-1c0e-49cd-aabe-409fc567bc48',
 14: '8da41d0f-5942-4aa2-9b35-5733cd864cee',
 15: '0839a18f-c733-4fed-8970-5c68c99389c4',
 16: '2478c6d8-697a-4e0f-98bd-7e35f304bcfd',
 17: 'f0415714-d076-4c6b-a23b-da81832a5360',
 18: 'b338a717-6ad2-4a6a-ad45-ff6033a97b15',
 19: 'ed4cba9e-1f16-41cf-9e3f-6e90c5b9095f',
 20: '1a4fa539-648f-4b6b-a090-934d4ba2e7ae',
 21: 'dca702ac-3b7c-4cb9-8a9d-93fa4d44d3a8',
 22: 'ccc48a30-89a7-

In [81]:
vector_store.get_by_ids(['1abdbbdb-a9db-477f-ae4b-61684bbc043f'])

[]

# Step 2 - Retrieval

In [82]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [83]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x783caa7aa7b0>, search_kwargs={'k': 4})

In [84]:
retriever.invoke('What is mmr?')

[Document(id='53dfa85c-cf1a-42d7-bc37-2f5219aa18de', metadata={}, page_content='यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर को कैसे इंप्लीमेंट कर सकते हो। सो यहां पे देखो गाइज़ यहां पे हमने क्या किया है? कुछ सैंपल डॉक्यूमेंट्स बना लिए हैं। सो यह लैंग चेन के ऊपर दो स्टेटमेंट है। उसके बाद क्रोमा के ऊपर है। उसके बाद एंबेडिंग के ऊपर है। फिर एमएमआर के ऊपर है। और फिर आगे फिर से लैंग चेन के ऊपर कुछ स्टेटमेंट्स हैं। ठीक है? और ये सारे डॉक्यूमेंट ऑब्जेक्ट्स हैं। उसके बाद हमने क्या किया? इस बार हमने लangचे डॉट कम्युनिटी वेक्टर'),
 Document(id='f4efbd92-e513-4de6-a0ac-0fd7007e5454', metadata={}, page_content='है उसको पिक करेगा। उसके बाद जो नेक्स्ट उसका पिक होगा वो एक ऐसा डॉक्यूमेंट होगा जो नॉट ओनली रेलेवेंट होगा बट पहले वाले से बहुत ज्यादा डिसिमिलर होगा। और ऐसा ही वह आगे करते चला जाता है और इस तरीके से वो डॉक्यूमेंट्स आपको फैच करके ला के देता है। तो दिस इज द मेन आइडियोलॉजी बिहाइंड एमएमआर कि नॉट ओनली आपको रेलेवेंट रिजल्ट्स आने चाहिए बट एट द से

# Step 3 - Augmentation

In [85]:
llm = llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

In [86]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.
      Always respond in the same language as the prompt.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [87]:
question          = "what is mmr"
retrieved_docs    = retriever.invoke(question)

In [88]:
retrieved_docs

[Document(id='53dfa85c-cf1a-42d7-bc37-2f5219aa18de', metadata={}, page_content='यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर को कैसे इंप्लीमेंट कर सकते हो। सो यहां पे देखो गाइज़ यहां पे हमने क्या किया है? कुछ सैंपल डॉक्यूमेंट्स बना लिए हैं। सो यह लैंग चेन के ऊपर दो स्टेटमेंट है। उसके बाद क्रोमा के ऊपर है। उसके बाद एंबेडिंग के ऊपर है। फिर एमएमआर के ऊपर है। और फिर आगे फिर से लैंग चेन के ऊपर कुछ स्टेटमेंट्स हैं। ठीक है? और ये सारे डॉक्यूमेंट ऑब्जेक्ट्स हैं। उसके बाद हमने क्या किया? इस बार हमने लangचे डॉट कम्युनिटी वेक्टर'),
 Document(id='f4efbd92-e513-4de6-a0ac-0fd7007e5454', metadata={}, page_content='है उसको पिक करेगा। उसके बाद जो नेक्स्ट उसका पिक होगा वो एक ऐसा डॉक्यूमेंट होगा जो नॉट ओनली रेलेवेंट होगा बट पहले वाले से बहुत ज्यादा डिसिमिलर होगा। और ऐसा ही वह आगे करते चला जाता है और इस तरीके से वो डॉक्यूमेंट्स आपको फैच करके ला के देता है। तो दिस इज द मेन आइडियोलॉजी बिहाइंड एमएमआर कि नॉट ओनली आपको रेलेवेंट रिजल्ट्स आने चाहिए बट एट द से

In [89]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर को कैसे इंप्लीमेंट कर सकते हो। सो यहां पे देखो गाइज़ यहां पे हमने क्या किया है? कुछ सैंपल डॉक्यूमेंट्स बना लिए हैं। सो यह लैंग चेन के ऊपर दो स्टेटमेंट है। उसके बाद क्रोमा के ऊपर है। उसके बाद एंबेडिंग के ऊपर है। फिर एमएमआर के ऊपर है। और फिर आगे फिर से लैंग चेन के ऊपर कुछ स्टेटमेंट्स हैं। ठीक है? और ये सारे डॉक्यूमेंट ऑब्जेक्ट्स हैं। उसके बाद हमने क्या किया? इस बार हमने लangचे डॉट कम्युनिटी वेक्टर\n\nहै उसको पिक करेगा। उसके बाद जो नेक्स्ट उसका पिक होगा वो एक ऐसा डॉक्यूमेंट होगा जो नॉट ओनली रेलेवेंट होगा बट पहले वाले से बहुत ज्यादा डिसिमिलर होगा। और ऐसा ही वह आगे करते चला जाता है और इस तरीके से वो डॉक्यूमेंट्स आपको फैच करके ला के देता है। तो दिस इज द मेन आइडियोलॉजी बिहाइंड एमएमआर कि नॉट ओनली आपको रेलेवेंट रिजल्ट्स आने चाहिए बट एट द सेम टाइम आपको डवर्स रिजल्ट्स मिलने चाहिए। ठीक है? तो नाउ दैट यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर\

In [90]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [91]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n      Always respond in the same language as the prompt. \n\n      यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर को कैसे इंप्लीमेंट कर सकते हो। सो यहां पे देखो गाइज़ यहां पे हमने क्या किया है? कुछ सैंपल डॉक्यूमेंट्स बना लिए हैं। सो यह लैंग चेन के ऊपर दो स्टेटमेंट है। उसके बाद क्रोमा के ऊपर है। उसके बाद एंबेडिंग के ऊपर है। फिर एमएमआर के ऊपर है। और फिर आगे फिर से लैंग चेन के ऊपर कुछ स्टेटमेंट्स हैं। ठीक है? और ये सारे डॉक्यूमेंट ऑब्जेक्ट्स हैं। उसके बाद हमने क्या किया? इस बार हमने लangचे डॉट कम्युनिटी वेक्टर\n\nहै उसको पिक करेगा। उसके बाद जो नेक्स्ट उसका पिक होगा वो एक ऐसा डॉक्यूमेंट होगा जो नॉट ओनली रेलेवेंट होगा बट पहले वाले से बहुत ज्यादा डिसिमिलर होगा। और ऐसा ही वह आगे करते चला जाता है और इस तरीके से वो डॉक्यूमेंट्स आपको फैच करके ला के देता है। तो दिस 

#Step 4 - Generation

In [92]:
answer = llm.invoke(final_prompt)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [93]:
print(answer.content)

[{'type': 'text', 'text': 'Based on the provided context, MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query. \n\nKey details about MMR from the context include:\n* **Goal/Ideology:** It aims to return results that are not only relevant to the search query but also diverse (different from each other).\n* **Avoids Redundancy:** Unlike regular similarity searches where retrieved documents might be very similar and repeat the same information, MMR avoids this to provide different perspectives.\n* **How it works:** It first picks the most relevant document. For the next pick, it chooses a document that is not only relevant to the query but also very dissimilar to the previously selected document, continuing this process to fetch the final results.', 'extras': {'signature': 'EqceCqQeARFNMg9ZOat4zX8bKunXxMpW3CjjhN13zA4WFfSy+s0JoHl4nMjmFkUhriHnTwBbI0/HpzVt0/ZdJ6/rUXgHoU4Dv5RnAKZUpaJeyXcTaa/v8QgbV2ki95g

#Building the Chain

In [94]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel , RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [95]:
def format_docs(retrieved_docs):
  context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [96]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [97]:
parallel_chain.invoke("what is mmr")

{'context': 'यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं कि आप एमएमआर को कैसे इंप्लीमेंट कर सकते हो। सो यहां पे देखो गाइज़ यहां पे हमने क्या किया है? कुछ सैंपल डॉक्यूमेंट्स बना लिए हैं। सो यह लैंग चेन के ऊपर दो स्टेटमेंट है। उसके बाद क्रोमा के ऊपर है। उसके बाद एंबेडिंग के ऊपर है। फिर एमएमआर के ऊपर है। और फिर आगे फिर से लैंग चेन के ऊपर कुछ स्टेटमेंट्स हैं। ठीक है? और ये सारे डॉक्यूमेंट ऑब्जेक्ट्स हैं। उसके बाद हमने क्या किया? इस बार हमने लangचे डॉट कम्युनिटी वेक्टर\n\nहै उसको पिक करेगा। उसके बाद जो नेक्स्ट उसका पिक होगा वो एक ऐसा डॉक्यूमेंट होगा जो नॉट ओनली रेलेवेंट होगा बट पहले वाले से बहुत ज्यादा डिसिमिलर होगा। और ऐसा ही वह आगे करते चला जाता है और इस तरीके से वो डॉक्यूमेंट्स आपको फैच करके ला के देता है। तो दिस इज द मेन आइडियोलॉजी बिहाइंड एमएमआर कि नॉट ओनली आपको रेलेवेंट रिजल्ट्स आने चाहिए बट एट द सेम टाइम आपको डवर्स रिजल्ट्स मिलने चाहिए। ठीक है? तो नाउ दैट यू अंडरस्टैंड कि एमएमआर होता क्या है? काम कैसे करता है? अब मैं आपको कोड में दिखाता हूं क

In [98]:
parser = StrOutputParser()

In [99]:
main_chain = parallel_chain | prompt | llm | parser

In [100]:
main_chain.invoke("we can retrieve the content from the vector store using semantic search then why we need Specific Retrievals?")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript context, we need specific retrievers because:\n\n1. **Single Strategy Limit of Vector Stores:** A vector store can only perform similarity search using a single strategy (comparing vectors based on a metric to return the top documents). It cannot perform a search using a different strategy on its own.\n2. **Advanced Search Strategies:** Retrievers allow you to use different and internally advanced search strategies to fetch relevant documents.\n3. **Performance in RAG Systems:** Different retrievers exist to solve specific smaller problems and are mainly used to build RAG-based systems to improve their performance.'